In [1]:
using LinearAlgebra

In [2]:
n = 64
d = 2
N = n^d

4096

In [ ]:
function grid_coords(n::Int, d::Int)
    N = n^d
    h = 1/(n+1)
    coords = Matrix{Float64}(undef, N, d)

    for k in 1:d
        inner = n^(k-1)
        period = n^k
        for i in 1:N
            coords[i, k] = (i-1) % period ÷ inner + 1
        end
    end

    return h * coords
end

grid_coords (generic function with 1 method)

In [13]:
# -- warm-up call --
# first run of the function compiles it
# run it first with some small args to trigger the compilation
# then run it again with the correct arguments
grid_coords(3,2)

9×2 Matrix{Float64}:
 0.25  0.25
 0.5   0.25
 0.75  0.25
 0.25  0.5
 0.5   0.5
 0.75  0.5
 0.25  0.75
 0.5   0.75
 0.75  0.75

In [5]:
@allocated grid_coords(n,d)

131248

In [6]:
using TimerOutputs
const to = TimerOutput()
@timeit to "grid" grid = grid_coords(n,d);
show(to)

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      308ms /   0.0%           63.4MiB /   0.2%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
grid           1   87.9μs  100.0%  87.9μs    128KiB  100.0%   128KiB
────────────────────────────────────────────────────────────────────

---

## lvl beginner

In [ ]:
function eval_f2(x, k, t)
    vec(prod(sin.(k' .* x), dims=2)) * exp(-t)
end
eval_f2(grid[1:4,:], ones(d), 0.0)

4-element Vector{Float64}:
 0.00023666771763934788
 0.00047327942035569606
 0.0007097791064837532
 0.0009461108008705071

In [ ]:
@allocated eval_f2(grid, ones(d), 0.0)

131432

In [ ]:
using TimerOutputs
const to = TimerOutput()
@timeit to "eval_f2" f2_eval = eval_f2(grid, ones(d), 0.0);
show(to)

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      901μs /   6.2%            150KiB /  85.9%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
eval_f2        1   55.8μs  100.0%  55.8μs    129KiB  100.0%   129KiB
────────────────────────────────────────────────────────────────────

---

## lvl pro

In [7]:
function eval_f(grid::Matrix{Float64}, k::Vector{Float64}, t::Float64)
    N, d = size(grid)
    out = ones(N)

    for j in 1:d
        kj = k[j]
        @inbounds for i in 1:N
            out[i] *= sin(kj * grid[i, j])
        end
    end

    exp_t = exp(-t)
    @inbounds for i in 1:N
        out[i] *= exp_t
    end

    return out
end
eval_f(grid[1:4,:], ones(d), 0.0)

4-element Vector{Float64}:
 0.00023666771763934788
 0.00047327942035569606
 0.0007097791064837532
 0.0009461108008705071

In [8]:
@allocated eval_f(grid, ones(d), 0.0)

32920

In [9]:
using TimerOutputs
const to = TimerOutput()
@timeit to "eval_f" f_eval = eval_f(grid, ones(d), 0.0);
show(to)

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      972μs /  10.0%           53.6KiB /  60.5%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
eval_f         1   97.4μs  100.0%  97.4μs   32.4KiB  100.0%  32.4KiB
────────────────────────────────────────────────────────────────────